In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, 
                             recall_score, f1_score, confusion_matrix)

# Load SMS spam dataset directly
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

print(df.shape)
print(df['label'].value_counts())
print(df.head())            

(5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [3]:
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
print(df['label_num'].value_counts())

label_num
0    4825
1     747
Name: count, dtype: int64


In [4]:
X = df['message']
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train)}")
print(f"Test size:  {len(X_test)}")
print(f"Spam ratio in train: {y_train.mean():.3f}")
print(f"Spam ratio in test:  {y_test.mean():.3f}")

Train size: 4457
Test size:  1115
Spam ratio in train: 0.134
Spam ratio in test:  0.134


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=3000, stop_words='english')

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f"Feature matrix shape: {X_train_tfidf.shape}")
print(f"Each message is now a vector of {X_train_tfidf.shape[1]} numbers")

Feature matrix shape: (4457, 3000)
Each message is now a vector of 3000 numbers


In [6]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

y_pred = lr_model.predict(X_test_tfidf)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1:        {f1_score(y_test, y_pred):.4f}")

Accuracy:  0.9722
Precision: 1.0000
Recall:    0.7919
F1:        0.8839


In [7]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(f"                 Predicted Ham  Predicted Spam")
print(f"Actual Ham       {cm[0][0]}            {cm[0][1]}")
print(f"Actual Spam      {cm[1][0]}            {cm[1][1]}")
print()
print(f"True Negatives  (ham correctly identified):   {cm[0][0]}")
print(f"False Positives (ham wrongly flagged spam):   {cm[0][1]}")
print(f"False Negatives (spam that slipped through):  {cm[1][0]}")
print(f"True Positives  (spam correctly caught):      {cm[1][1]}")

Confusion Matrix:
                 Predicted Ham  Predicted Spam
Actual Ham       966            0
Actual Spam      31            118

True Negatives  (ham correctly identified):   966
False Positives (ham wrongly flagged spam):   0
False Negatives (spam that slipped through):  31
True Positives  (spam correctly caught):      118


In [8]:
# Get probabilities instead of hard predictions
y_proba = lr_model.predict_proba(X_test_tfidf)[:, 1]

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 48)
for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    p = precision_score(y_test, y_pred_t)
    r = recall_score(y_test, y_pred_t)
    f = f1_score(y_test, y_pred_t)
    print(f"{t:<12} {p:<12.4f} {r:<12.4f} {f:<12.4f}")

Threshold    Precision    Recall       F1          
------------------------------------------------
0.3          0.9552       0.8591       0.9046      
0.4          0.9922       0.8591       0.9209      
0.5          1.0000       0.7919       0.8839      
0.6          1.0000       0.6711       0.8032      
0.7          1.0000       0.5168       0.6814      


In [9]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(lr_model,   '../models/spam_classifier.pkl')
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
print("Saved spam classifier and vectorizer")

Saved spam classifier and vectorizer
